# 18.3 Nonlinear expressions

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Any of AMPL's arithmetic operators ([Table 7-1](../07/7_2_arithmetic_expressions.ipynb#table-7-1)) and arithmetic functions ([Table 7-2](../07/7_2_arithmetic_expressions.ipynb#table-7-2))
may be applied to variables as well as parameters. If any resulting objective or constraint
does not satisfy the rules for linearity ([Chapter 8](../08/08.md)) or piecewise-linearity ([Chapter 17](../17/17.md)),
AMPL treats it as "not linear". When you type `solve`, AMPL passes along instructions
that are sufficient for your solver to evaluate every expression in every objective and
constraint, together with derivatives if appropriate.

If you are using a typical nonlinear solver, it is up to you to define your objective and
constraints in terms of the "smooth" functions that the solver requires. The generality of
AMPL's expression syntax can be misleading in this regard. For example, if you are trying
to use variables Flow[i,j] representing flow between points i and j, it is tempting
to write expressions like

```ampl
cost[i,j] * abs(Flow[i,j])
```

or

```ampl
if Flow[i,j] = 0 then 0 else base[i,j] + cost[i,j]*Flow[i,j]
```

These are certainly not linear, but the first is not smooth (its slope changes abruptly at
zero) and the second is not even continuous (its value jumps suddenly at zero). If you try
to use such expressions, AMPL will not complain, and your solver may even return what
it claims to be an optimal solution — but the results could be wrong.

Expressions that apply nonsmooth functions (such as abs, min, and max) to variables
generally produce nonsmooth results; the same is true of if-then-else expressions
in which a condition involving variables follows if. Nevertheless, there are useful
exceptions where a carefully written expression can preserve smoothness. As an exam-

![](18_3_smooth_nonlinear_functions.png)

<a id='fig-18-3'><center><b>Figure 18-3:</b> Smooth nonlinear functions.</center></a>


**[Figure 18-3](./18_3_nonlinear_expressions.ipynb#fig-18-3):** Smooth nonlinear functions.

ple, consider again the flow-pressure relationship from [Section 18.1](./18_1_sources_of_nonlinearity.ipynb). If the pressure is
greater at city i than at city j, then the flow is from i to j and is related to the pressure by

```ampl
Flow[i,j]ˆ2 = c[i,j]ˆ2 * (Press[i]ˆ2 - Press[j]ˆ2)
```

If instead the pressure is greater at city `j` than at city i, a similar equation can be written:

```ampl
Flow[j,i]ˆ2 = c[j,i]ˆ2 * (Press[j]ˆ2 - Press[i]ˆ2)
```

But since the constants c[i,j] and c[j,i] refer to the same pipe, they are equal.
Thus instead of defining a separate variable for flow in each direction, we can let
Flow[i,j] be unrestricted in sign, with a positive value indicating flow from i to j
and a negative value indicating the opposite. Using this variable, the previous pair of
flow-pressure constraints can be replaced by one:

```ampl
(if Flow[i,j] >= 0 then Flow[i,j]ˆ2 else -Flow[i,j]ˆ2)
       = c[i,j]ˆ2 * (Press[i]ˆ2 - Press[j]ˆ2)
```

Normally the use of an if expression would give rise to a nonsmooth constraint, but in
this case it gives a function whose two quadratic halves "meet" smoothly where
Flow[i,j] is zero, as seen in [Figure 18-3a](./18_3_nonlinear_expressions.ipynb#fig-18-3a).

As another example, the convex function in [Figure 18-3b](./18_3_nonlinear_expressions.ipynb#fig-18-3b) is most easily written
log(Flow[i,j]) / (Flow[i,j]-1), but unfortunately if Flow[i,j] is 1 this
simplifies to 0/0, which would be reported as an error. In fact, this expression does not
evaluate accurately if Flow[i,j] is merely very close to zero. If instead we write

```ampl
if abs(Flow[i,j]-1) > 0.00001 then
       log(Flow[i,j])/(Flow[i,j]-1)
else
       1.5 - Flow[i,j]/2
```

a highly accurate linear approximation is substituted at small magnitudes of
Flow[i,j]. This alternative is not smooth in a literal mathematical sense, but it is
numerically close enough to being smooth to suffice for use with some solvers.

In the problem instance that it sends to a solver, AMPL distinguishes linear from nonlinear
constraints and objectives, and breaks each constraint and objective expression into
a sum of linear terms plus a not-linear part. Additional terms that become linear due to
the fixing of some variables are recognized as linear. For example, in our example from
[Section 18.1](./18_1_sources_of_nonlinearity.ipynb),

```ampl
minimize Total_Cost:
       sum {i in ORIG, j in DEST} Cost[i,j] * Ship[i,j];
```

each fixing of a Cost[i,j] variable gives rise to a linear term; if all the Cost[i,j]
variables are fixed, then the objective is represented to the solver as linear. Variables
may be fixed by a `fix` command ([Section 11.4](../11/11_4_modifying_models.ipynb)) or through the actions of the presolve
phase ([Section 14.1](../14/14_1_presolve.ipynb)); although the presolving algorithm ignores nonlinear constraints, it
works with any linear constraints that are available, as well as any constraints that
become linear as variables are fixed.

AMPL's built-in functions are only some of the ones most commonly used in model
formulations. Libraries of other useful functions can be introduced when needed. To use
cumulative normal and inverse cumulative normal functions from a library called
statlib, for example, you would first load the library with a statement such as

```ampl
load statlib.dll;
```

and declare the functions by use of AMPL function statements:

```ampl
function cumnormal;
function invcumnormal;
```

Your model could then make use of these functions to form expressions such as
cumnormal(mean[i],sdev[i],Inv[i,t]) and invcumnormal(6). If these
functions are applied to variables, AMPL also arranges for function evaluations to be carried
out during the solution process.

A function declaration specifies a library function's name and (optionally) its
required arguments. There may be any number of arguments, and even iterated collections
of arguments. Each function's declaration must appear before its use. For your
convenience, a script containing the function declarations may be supplied along with
the library, so that a statement such as include statlib is sufficient to provide
access to all of the library's functions. Documentation for the library will indicate the
functions available and the numbers and meanings of their arguments.
Determining the correct load command may involve a number of details that depend
on the type of system you're using and even its specific configuration. See [Section A.22](#missing) <!--- xTODO missing reference --->
for further discussion of the possibilities and the related AMPLFUNC option.

If you are ambitious, you can write and compile your own function libraries. Instructions
and examples are available from the AMPL web site.